# 3.2.2 MIND 多兴趣召回

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

当用户同时喜欢科幻、跑步和烹饪时，为什么一个平均向量会丢失兴趣？如何用多个向量分别检索？

## Setup

本 Notebook 的默认真实数据是 **Amazon Books 2014：按 MIND 论文执行迭代 10-core，并核验 6,271,511 行**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [Li et al., 2019, MIND](https://arxiv.org/abs/1904.08030)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "mind-amazon-books"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "root": str(ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 学习地图

1. 从原始论文理解系统约束；
2. 用可手算数字读懂公式和形状；
3. 检查数据、切分与标签；
4. 使用工业框架模型类训练；
5. 分开验证训练、推理和测试；
6. 用实际输出讨论失败边界。

**本节问题：** 当用户同时喜欢科幻、跑步和烹饪时，为什么一个平均向量会丢失兴趣？如何用多个向量分别检索？

**先修知识：** 3.0 的向量、概率与损失函数。第一次阅读无需推导梯度，只要能解释输入、输出和形状。

## Paper & Context

MIND 用动态路由把行为序列聚成多个兴趣胶囊，训练时用 label-aware attention 选择与目标物品最相关的兴趣。关键变化是单用户单向量变成单用户多向量；代价是多路 ANN、去重和兴趣数选择。

**来源：** [Li et al., 2019, MIND](https://arxiv.org/abs/1904.08030)

### 原文实验设计与关键结论

原文在 Amazon Books 与工业数据上用 HitRate@10/50/100 比较单兴趣与多兴趣方法。full 档恢复论文对应的 2014 Amazon release、迭代 10-core 与 Table 1 统计；统计不匹配会直接终止，而不会生成伪复现结果。

请区分三层证据：论文中的离线实验、本 Notebook 验证的代码链路、生产系统尚需验证的在线收益。三者不能互相替代。

## Reproduction Contract

**正式数据：** Amazon Books ratings-only 2014, iterative 10-core  
**资源 ID：** `amazon-books-2014-ratings`  
**切分：** paper protocol: random 19:1 train/test; a deterministic seed records the sampled target  
**指标：** HitRate@10, HitRate@50, HitRate@100  
**与论文比较边界：** MIND Table 1/2: 351,356 users, 393,801 goods, 6,271,511 samples; absolute comparison is accepted only if these post-filter counts match

`full` 是论文对照唯一有效的效果模式：它不截断用户、物品或测试行。`smoke` 只做张量、损失和推理链路回归，不进入论文数值比较。

## Model Structure & Formula Walkthrough

![Figure 2 · MIND overview](/static/paper-figures/mind.webp)

> **论文原图节选** · Figure 2 · MIND overview · PDF p.3。下图直接截取自原文，用于对照下方公式与代码。

### 关键模块

- **行为 Embedding + Pooling**：把每次点击的物品 id / 类别 embedding 平均成行为向量，而不是一开始就压成一个用户向量。
- **动态路由胶囊**：多轮“投票”：相似行为反复把票投给同一个兴趣胶囊，最终得到 K 个兴趣向量（而不是 1 个）。
- **Label-aware Attention**：训练时用目标物品做 query、从 K 个兴趣中选最相关的一个；推理时让 K 个兴趣各自召回再合并。

### 结构：一段历史变成多个兴趣向量

历史商品 embedding 组成 $H\in\mathbb R^{B\times L\times d}$。动态路由维护行为 $j$ 到兴趣胶囊 $k$ 的权重 $c_{jk}=\mathrm{softmax}_k(b_{jk})$，先加权汇总 $s_k=\sum_jc_{jk}h_j$，再用 squash 保留方向并限制长度：

$$v_k=\frac{\|s_k\|^2}{1+\|s_k\|^2}\frac{s_k}{\|s_k\|},\qquad b_{jk}\leftarrow b_{jk}+h_j^\top v_k.$$

相似的行为会逐轮增加对同一胶囊的赞成票。训练时目标商品 $e_i$ 选择最相关兴趣，常写成 $a_k\propto\exp((v_k^\top e_i)^p)$，再令 $v_u=\sum_ka_kv_k$ 并使用 sampled-softmax。推理时不知道目标，因此让 $K$ 个兴趣分别 ANN 检索后合并去重。

```text
history [B,L,d] -> routing -> interests [B,K,d] -> K-way ANN -> merge
```

### 公式到代码

`run_mind` 把真实时间序列整理为定长张量；Torch-RecHub 的 CapsuleNetwork 完成路由，训练标签只用于选择兴趣而不会进入线上用户特征。

阅读源码时按“张量形状 → 前向计算 → score → loss → metric”五步追踪，不需要一次读完整个工程文件。

## Math by Hand

设用户有 $K$ 个兴趣向量 $V_u=[v_1,\ldots,v_K]$。候选物品 $e_i$ 的分数取 $\max_k v_k^\top e_i$。可先把历史点分成两团，各自平均得到两个质心；若强行平均成一个点，它可能落在两团之间，反而不像任何真实兴趣。

下面用 NumPy/Matplotlib 验证直觉。二维图只是教学投影，工业 embedding 虽有更多维，计算规则相同。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
history=np.array([[1,.1],[.9,.2],[.1,1],[.2,.9]])
a,b,single=history[:2].mean(0),history[2:].mean(0),history.mean(0)
fig,ax=plt.subplots(figsize=(5.5,4.4)); ax.scatter(history[:,0],history[:,1],s=90,label='history')
ax.scatter(*a,s=180,marker='*',label='interest 1'); ax.scatter(*b,s=180,marker='*',label='interest 2')
ax.scatter(*single,s=120,marker='x',label='single average'); ax.set(title='Two interests avoid mixed intent',xlabel='topic A',ylabel='topic B')
ax.legend(); ax.grid(alpha=.2); plt.show()
print({'interest_1':a.tolist(),'interest_2':b.tolist(),'single':single.tolist()})

## Data

full 档从论文对应的 Amazon Books 2014 ratings-only 全量文件出发，反复执行 user/item ≥10 过滤，并强制核验论文 Table 1 的 351,356 用户、393,801 商品、6,271,511 行；smoke 档才使用仓库内真实小数据验证张量链路。

**防泄漏清单：**按时间切分；item 映射只表达已知目录，不读取测试标签；低评分或未点击负反馈均来自数据中的已观察行；序列只看预测时刻以前；测试集只在最后评价。CPU 档使用真实数据的确定性子集，**不是统一 benchmark 成绩**。

## Model & Framework

实际使用 torch_rechub.models.matching.MIND，执行 CapsuleNetwork、目标兴趣选择和多兴趣推理；full profile 映射 TorchEasyRec MIND。

smoke 档强调模型类、张量契约和指标链路真实可运行；full 档应替换原始数据、分布式配置、索引/服务和资源预算，而不是只增加 epoch。

In [ ]:
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from importlib import import_module
from recsys_lab.runtime import save_records

# 算法实现就在当前章节目录，不再通过公共模块隐藏。
chapter_train = import_module("chapter_code.3_2_2_mind.train")
run_mind = chapter_train.run_mind

print("实际执行函数源码（包含数据、训练、推理和测试）：")
print(inspect.getsource(run_mind))

## Train & Inference

下一格固定 seed、构造数据、实例化模型、训练并进入推理路径。生成式章节在 CUDA 上执行完整评测；CPU 环境只验证缩小后的基本张量与约束链路。

In [ ]:
result = run_mind()
print({'framework': result['framework'], 'dataset': result.get('dataset', {}),
       'device': result.get('device'), 'validation_mode': result.get('validation_mode')})
print('inference contract:', 'user tower 输出 [B,K,d]；每个兴趣独立检索，再按最高分合并、去重和配额控制。')
assert np.isfinite(result['loss_curve']).all()
print('loss:', round(result['loss_curve'][0],4), '→', round(result['loss_curve'][-1],4))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10.5,3.5))
axes[0].plot(result['loss_curve'],color='#4f8f00',lw=2); axes[0].set(title='Training loss',xlabel='epoch',ylabel='loss'); axes[0].grid(alpha=.2)
metrics={'recall@10': result['recall@10'], 'positive_top1': result['positive_top1']}
axes[1].bar(range(len(metrics)),list(metrics.values()),color=['#7ca832','#e1874b','#6d88a4'][:len(metrics)])
axes[1].set_xticks(range(len(metrics)),list(metrics),rotation=18); axes[1].set(title='Executed test metrics',ylim=(0,max(1.0,max(metrics.values())*1.15)))
for index,value in enumerate(metrics.values()): axes[1].text(index,value,f'{value:.3f}',ha='center',va='bottom')
plt.tight_layout(); plt.show(); display(pd.Series(metrics,name='value').to_frame())

In [ ]:
# 论文数字只能在数据、切分、候选和指标全部同口径时相减。
paper_protocol = json.loads('{"dataset": "Amazon Books ratings-only 2014, iterative 10-core", "resource": "amazon-books-2014-ratings", "split": "paper protocol: random 19:1 train/test; a deterministic seed records the sampled target", "filters": "repeat user/item >=10 filtering to convergence", "metrics": ["HitRate@10", "HitRate@50", "HitRate@100"], "paper_targets": {"HitRate@10": 0.0309, "HitRate@50": 0.1101, "HitRate@100": 0.1631}, "paper_comparison": "MIND Table 1/2: 351,356 users, 393,801 goods, 6,271,511 samples; absolute comparison is accepted only if these post-filter counts match"}')
paper_targets = paper_protocol.get('paper_targets', {})
metric_key = {'HitRate@10':'paper_protocol_hr@10', 'NDCG@10':'paper_protocol_ndcg@10',
              'AUC':'auc', 'LogLoss':'logloss'}
dataset_name = result.get('dataset', {}).get('dataset', '')
dataset_aligned = paper_protocol.get('dataset', '').split(',')[0].casefold() in dataset_name.casefold()
comparison_eligible = PROFILE == 'full' and dataset_aligned
rows=[]
for paper_metric,target in paper_targets.items():
    result_key=metric_key.get(paper_metric)
    value=result.get(result_key) if result_key else None
    rows.append({'metric':paper_metric,'tutorial':value,'paper':target,
                 'absolute_gap':None if value is None or not comparison_eligible else float(value)-float(target),
                 'comparable':comparison_eligible and value is not None})
if rows:
    display(pd.DataFrame(rows))
    if not comparison_eligible:
        print('NOT COMPARABLE：当前运行的数据/协议与论文不完全一致，不计算复现差值。')
else:
    print('论文没有可公开、可同口径复现的绝对目标；本节只报告结构与公开协议验证。')

## Test & Results Discussion

In [ ]:
display(Markdown(f'''### 本次已执行结果

- 主指标 recall@10 = **{result['recall@10']:.4f}**。
- 辅助指标 positive_top1 = **{result['positive_top1']:.4f}**。
- 本节没有把不同任务的数值伪装成 baseline；章节总结只做同口径比较。
- 训练损失从 **{result['loss_curve'][0]:.4f}** 降到 **{result['loss_curve'][-1]:.4f}**。损失下降只说明优化工作，不等于泛化或业务收益。
- **结果解释：** 只有 full 档同时满足论文数据版本、10-core 后统计、K=3、d=36 和候选协议时才允许对照 Table 2；smoke 指标只证明代码可运行。

### 工业边界

user tower 输出 [B,K,d]；每个兴趣独立检索，再按最高分合并、去重和配额控制。

上线前还需验证延迟、吞吐、内存/显存、数据新鲜度、校准、回滚和线上 A/B。
'''))

In [ ]:
record={
    'algorithm': 'MIND 多兴趣召回',
    'primary_metric': 'recall@10', 'primary_value': float(result['recall@10']),
    'secondary_metric': 'positive_top1', 'secondary_value': float(result['positive_top1']),
    'baseline_metric': None,
    'baseline_value': float(result[None]) if False else None,
    'framework': result['framework'], 'source_notebook': '3_2_2_mind',
    'validation_mode': result.get('validation_mode', 'standard'),
    'dataset': result['dataset']['dataset'],
    'randomly_fabricated_rows': int(result['dataset']['randomly_fabricated_rows'])
}
path=save_records('chapter_3_2','3_2_2_mind',[record]); print('saved:',path.relative_to(ROOT))

## Checks

自动断言用于防止数据、训练和指标链路静默失效，不是效果证明。

In [ ]:
assert result['loss_curve'][-1] < result['loss_curve'][0]
assert 0 <= float(result['recall@10']) <= 1
assert np.isfinite(float(result['positive_top1']))
print('PASS：数据、训练、推理、测试和结果产物均已验证。')

## Next Steps

1. 换成对应公开数据的完整时间切分；2. 增加强 baseline 与消融；3. 记录效果、延迟和成本；4. 映射到 TorchEasyRec/官方 full profile；5. 只在相同候选与数据口径下比较。